# 08b_screen_libraries_repr — 재설계 모델로 3개 DB 재스크리닝 (신규)

**한 줄 요약:** 23의 **표현 기반 앙상블**(재균형·class_weight)로 FooDB·NPASS·COCONUT를 다시 예측한다.
**이전(08)과 차이:** 08은 지문 6종 concat + LightGBM 하나. 여기선 **상위 (모델×표현) 5조합**(maccs/rdkit/topotorsion 지문 + 2D descriptor)을 소프트보팅.
**관전 포인트:** GW-4064 같은 합성 오탐이 줄어드는지, 후보 수·순위가 어떻게 바뀌는지.
**큰 흐름:** ① 준비 → ② 모델·표현계산기 → ③ DB목록 → ④ 청크 스크리닝 → ⑤ 저장

> **📌 읽는 법**: 각 코드 셀은 [① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기].

### 준비 — 도구 불러오기

In [ ]:
import os
while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')  # data/ 폴더를 찾을 때까지 상위로 (하위 폴더에서 열어도 동작)
print('작업 폴더:', os.getcwd())
import time, pickle
import numpy as np, pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator, MACCSkeys, Descriptors
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

🔎 **코드 뜯어보기 (준비)**
- `Descriptors`=2D descriptor 계산(desc2d 표현에 필요). 나머지는 04/08에서 설명.

### 셀 1 — 표현앙상블 + 표현별 계산기
상위 조합·학습된 파이프라인을 불러오고, 각 표현(maccs/rdkit/topotorsion/desc2d)을 학습과 같은 순서로 만드는 함수.

In [ ]:
# 23이 저장한 표현앙상블 로드 + 표현별 계산기(학습과 동일 순서) 준비
with open("data/HSD17B13_repr_ensemble.pkl", "rb") as f:
    B = pickle.load(f)
TOP   = B["top"]                              # [[모델,표현], ...] 상위 조합
REPS  = B["reps"]                             # 표현이름 -> 열목록(순서 중요)
PIPES = B["pipelines"]                        # "모델|표현" -> 학습된 파이프라인
NB = 1024
D2_NAMES = REPS["desc2d"]                     # 필요한 2D descriptor 이름 36개(학습 순서)
D2_FUNCS = [getattr(Descriptors, n) for n in D2_NAMES]  # 36개 '개별' 계산 함수(전체 217개 계산 회피=속도 13배)
print("표현앙상블 로드 | 상위조합:", TOP)

gen_rdk = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=NB)
gen_tt  = rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=NB)
def _bits(fp, n):
    a = np.zeros((n,), dtype=np.int8); DataStructs.ConvertToNumpyArray(fp, a); return a

def rep_matrix(mols, rep):
    # 학습과 '완전히 같은 순서'로 각 표현을 계산해야 파이프라인이 올바로 예측
    if rep == "maccs":
        return np.vstack([_bits(MACCSkeys.GenMACCSKeys(m), 167) for m in mols]).astype(np.float32)
    if rep == "rdkit":
        return np.vstack([gen_rdk.GetFingerprintAsNumPy(m) for m in mols]).astype(np.float32)
    if rep == "topotorsion":
        return np.vstack([gen_tt.GetFingerprintAsNumPy(m) for m in mols]).astype(np.float32)
    if rep == "desc2d":
        # 필요한 36개 descriptor만 개별 함수로 계산(전체 217개 계산 회피 → 13배 빠름)
        arr = np.array([[fn(m) for fn in D2_FUNCS] for m in mols], dtype=np.float64)
        arr[~np.isfinite(arr)] = np.nan                    # inf/-inf → NaN (imputer가 채우도록)
        return arr
    raise ValueError(rep)

def predict_proba(mols):
    # 상위 조합마다 자기 표현으로 예측 → 확률 평균(소프트보팅). 표현은 한 번만 계산해 재사용
    need = sorted(set(r for _, r in TOP))
    mats = {r: rep_matrix(mols, r) for r in need}
    ps = []
    for mn, rn in TOP:
        ps.append(PIPES[f"{mn}|{rn}"].predict_proba(mats[rn])[:, 1])
    return np.mean(ps, axis=0)

🔎 **코드 뜯어보기 (셀 1)**
- `rep_matrix(mols, rep)` : 표현 종류별로 지문/descriptor 행렬 생성. desc2d는 `CalcMolDescriptors`로 전체 계산 후 **학습에 쓴 36개만 그 순서대로** 추출.
- `predict_proba(mols)` : 각 상위 조합이 자기 표현으로 예측한 확률을 `np.mean`으로 평균(소프트보팅). 표현은 한 번만 계산해 재사용.

### 셀 2 — 3개 DB 목록

In [ ]:
# 스크리닝할 3개 천연물 DB (08과 동일)
LIBS = [
    ("FooDB",   "data/foodb_2020_4_7_csv/foodb_2020_04_07_csv/Compound.csv", "cas_number",       "public_id", ","),
    ("NPASS",   "data/npass_structures.tsv",                                 "SMILES",           "np_id",     "\t"),
    ("COCONUT", "data/coconut_csv-09-2026.csv",                              "canonical_smiles", "identifier","," ),
]
PROB_MIN = 0.5
CHUNK = 5000

🔎 **코드 뜯어보기 (셀 2)**
- 08과 동일. `PROB_MIN=0.5` 이상만 저장, `CHUNK=5000`씩 처리.

### 셀 3 — 청크 스크리닝

In [ ]:
# 청크로 읽어 표현앙상블 예측. prob>=PROB_MIN 후보만 모음
def screen(name, path, smi_col, id_col, sep):
    hits, n_seen, t0 = [], 0, time.time()
    for chunk in pd.read_csv(path, sep=sep, usecols=[smi_col, id_col],
                             chunksize=CHUNK, on_bad_lines="skip", low_memory=False):
        ids, mols = [], []
        for cid, smi in zip(chunk[id_col], chunk[smi_col]):
            m = Chem.MolFromSmiles(str(smi))
            if m is not None:
                ids.append(cid); mols.append(m)
        n_seen += len(chunk)
        if not mols:
            continue
        prob = predict_proba(mols)
        for cid, m, p in zip(ids, mols, prob):
            if p >= PROB_MIN:
                hits.append((name, cid, Chem.MolToSmiles(m), float(p)))
        if n_seen % (CHUNK * 10) == 0:
            print(f"  [{name}] {n_seen} 처리, hit {len(hits)} ({time.time()-t0:.0f}s)")
    print(f"[{name}] 완료: {n_seen} 중 prob>={PROB_MIN} 후보 {len(hits)}개 ({time.time()-t0:.0f}s)")
    return hits

all_hits = []
for name, path, sc, ic, sep in LIBS:
    if not os.path.exists(path):
        print(f"[{name}] 파일 없음 → 건너뜀:", path); continue
    all_hits += screen(name, path, sc, ic, sep)

🔎 **코드 뜯어보기 (셀 3)**
- `predict_proba(mols)`로 청크를 한 번에 예측. desc2d 계산이 있어 08보다 다소 느림. prob≥0.5만 `hits`에 저장.

### 셀 4 — 결과 저장

In [ ]:
# 결과 저장 + DB별 개수/상위
res = pd.DataFrame(all_hits, columns=["source", "id", "canonical_smiles", "active_prob"])
res = res.sort_values("active_prob", ascending=False).reset_index(drop=True)
res.to_csv("data/screen_3db_hits_repr.csv", index=False)
print("\n총 후보(prob>=%.2f): %d개" % (PROB_MIN, len(res)))
print(res.groupby("source").size().to_string())
print("\n=== 전체 상위 15 ===")
print(res.head(15).to_string(index=False))
with pd.ExcelWriter("data/screen_3db_hits_repr.xlsx", engine="openpyxl") as w:
    res.head(1000).to_excel(w, sheet_name="top1000", index=False)
    for s in res["source"].unique():
        res[res.source == s].head(300).to_excel(w, sheet_name=f"{s}_top300", index=False)
print("저장: data/screen_3db_hits_repr.csv , data/screen_3db_hits_repr.xlsx")

🔎 **코드 뜯어보기 (셀 4)**
- `_repr` 접미사 파일로 저장(08 결과와 구분). DB별 개수·상위 15 출력.